# ISIN to Market Data Pipeline

This notebook creates a reusable mapping from the fund-disclosure universe to NSE symbols and optionally downloads historical prices.

Pipeline stages:
1. Load and validate the `INE`-only security universe.
2. Download NSE security masters.
3. Normalize and combine listings.
4. Map ISINs to symbols and report unresolved securities.
5. Optionally download daily Yahoo Finance history for mapped symbols.

Outputs are written to `data/market/isin_pipeline`.

In [ ]:
from pathlib import Path
import io
import re
import time
import warnings

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

UNIVERSE_PATH = PROJECT_ROOT / "data/processed/security_universe_INE_only.xlsx"
REFERENCE_DATA_DIR = PROJECT_ROOT / "data/raw/reference_market_data"
CPI_PATH = REFERENCE_DATA_DIR / "india_cpi_dataset.xlsx"
NSE_EQUITY_PATH = REFERENCE_DATA_DIR / "nse_equity_security_master.csv"
OUTPUT_DIR = PROJECT_ROOT / "data/market/isin_pipeline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NSE_SOURCES = {
    "EQUITY": "https://archives.nseindia.com/content/equities/EQUITY_L.csv",
    "SME": "https://nsearchives.nseindia.com/emerge/corporates/content/SME_EQUITY_L.csv",
    "ETF": "https://nsearchives.nseindia.com/content/equities/eq_etfseclist.csv",
    "MF": "https://nsearchives.nseindia.com/content/equities/mf_close-end.csv",
    "DEBT": "https://nsearchives.nseindia.com/content/equities/DEBT.csv",
}

print("Project root:", PROJECT_ROOT)
print("Universe:", UNIVERSE_PATH)
print("CPI reference:", CPI_PATH)
print("NSE equity reference:", NSE_EQUITY_PATH)
print("Output directory:", OUTPUT_DIR)

## 1. Load the security universe

The input workbook is expected to contain an `ISIN` column. Only identifiers beginning with `INE` are retained, deduplicated, and normalized to uppercase.

In [ ]:
def load_universe(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Universe file not found: {path}")

    universe = pd.read_excel(path, sheet_name="Universe")
    universe.columns = [str(column).strip() for column in universe.columns]
    if "ISIN" not in universe.columns:
        raise ValueError("The Universe sheet must contain an ISIN column.")

    universe["ISIN"] = (
        universe["ISIN"].astype(str).str.strip().str.upper()
    )
    universe = universe[universe["ISIN"].str.startswith("INE")].copy()
    universe = universe.drop_duplicates(subset=["ISIN"]).reset_index(drop=True)
    return universe

universe = load_universe(UNIVERSE_PATH)
print(f"Universe rows: {len(universe):,}")
display(universe.head())

## 2. Download and normalize NSE reference data

Column names differ across NSE files, so the normalizer locates ISIN, symbol, name, and series columns by aliases. Failed sources are reported and do not stop the other downloads.

In [ ]:
def make_session() -> requests.Session:
    session = requests.Session()
    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 Chrome/120 Safari/537.36"
        ),
        "Accept": "*/*",
        "Referer": "https://www.nseindia.com/",
    })
    retry = Retry(total=2, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
    session.mount("https://", HTTPAdapter(max_retries=retry))
    return session

def find_column(columns, aliases):
    normalized = {re.sub(r"[^A-Z0-9]", "", str(column).upper()): column for column in columns}
    for alias in aliases:
        key = re.sub(r"[^A-Z0-9]", "", alias.upper())
        if key in normalized:
            return normalized[key]
    return None

def normalize_nse_data(frame: pd.DataFrame, segment: str) -> pd.DataFrame:
    isin_col = find_column(frame.columns, ["ISIN", "ISIN_CODE", "ISIN NUMBER", "ISINNO"])
    symbol_col = find_column(frame.columns, ["SYMBOL", "SECURITY_SYMBOL", "SCRIP", "TICKER"])
    name_col = find_column(frame.columns, ["NAME OF COMPANY", "COMPANY NAME", "NAME", "SECURITY NAME"])
    series_col = find_column(frame.columns, ["SERIES", "SERIES CODE"])
    if isin_col is None:
        return pd.DataFrame(columns=["ISIN", "SYMBOL", "NAME", "SERIES", "SEGMENT"])

    result = pd.DataFrame({"ISIN": frame[isin_col].astype(str).str.strip().str.upper()})
    result["SYMBOL"] = frame[symbol_col].astype(str).str.strip() if symbol_col else pd.NA
    result["NAME"] = frame[name_col].astype(str).str.strip() if name_col else pd.NA
    result["SERIES"] = frame[series_col].astype(str).str.strip() if series_col else pd.NA
    result["SEGMENT"] = segment
    result = result[
        result["ISIN"].str.startswith("INE")
        & result["ISIN"].ne("INENAN")
    ]
    return result.drop_duplicates()

def download_nse_master(sources: dict) -> pd.DataFrame:
    session = make_session()
    frames = []
    for segment, url in sources.items():
        try:
            if segment == "EQUITY" and NSE_EQUITY_PATH.exists():
                frame = pd.read_csv(NSE_EQUITY_PATH)
                print(f"{segment}: loaded local reference file")
            else:
                response = session.get(url, timeout=45)
                response.raise_for_status()
                frame = pd.read_csv(io.StringIO(response.text))
            normalized = normalize_nse_data(frame, segment)
            if not normalized.empty:
                frames.append(normalized)
            print(f"{segment}: {len(normalized):,} rows")
        except Exception as error:
            print(f"{segment}: skipped ({error})")

    if not frames:
        raise RuntimeError("No NSE reference data was downloaded.")
    return pd.concat(frames, ignore_index=True).drop_duplicates()

nse_master = download_nse_master(NSE_SOURCES)
nse_master.to_csv(OUTPUT_DIR / "nse_reference_master.csv", index=False)
print(f"Combined NSE rows: {len(nse_master):,}")
display(nse_master.head())

## 3. Build the ISIN-to-symbol mapping

The mapping retains all distinct ISIN/symbol combinations in the audit file and selects one preferred symbol per ISIN for downstream price downloads.

In [ ]:
segment_priority = {"EQUITY": 0, "SME": 1, "ETF": 2, "MF": 3, "DEBT": 4}
nse_master["priority"] = nse_master["SEGMENT"].map(segment_priority).fillna(99)

all_matches = universe.merge(nse_master, on="ISIN", how="left")
all_matches = all_matches.sort_values(["ISIN", "priority", "SYMBOL"], na_position="last")

preferred = (
    all_matches[all_matches["SYMBOL"].notna() & all_matches["SYMBOL"].ne("nan")]
    .drop_duplicates("ISIN")
    .assign(YAHOO_SYMBOL=lambda frame: frame["SYMBOL"].str.upper() + ".NS")
)

unmapped = universe[~universe["ISIN"].isin(preferred["ISIN"])].copy()
preferred.to_csv(OUTPUT_DIR / "isin_to_symbol.csv", index=False)
unmapped.to_csv(OUTPUT_DIR / "unmapped_isins.csv", index=False)

print(f"Total universe ISINs: {len(universe):,}")
print(f"Mapped ISINs: {len(preferred):,}")
print(f"Unmapped ISINs: {len(unmapped):,}")
print(f"Mapping coverage: {len(preferred) / len(universe):.2%}")
display(preferred[["ISIN", "SYMBOL", "NAME", "SEGMENT", "YAHOO_SYMBOL"]].head())

## 4. Optional historical prices

Run this section only after installing `yfinance`. It downloads daily history for the preferred NSE symbols and writes a long-format Parquet file suitable for feature engineering.

In [ ]:
# Install once in the active environment if needed:
# %pip install yfinance

DOWNLOAD_HISTORY = False
START_DATE = "2010-01-01"
END_DATE = pd.Timestamp.today().strftime("%Y-%m-%d")

if DOWNLOAD_HISTORY:
    import yfinance as yf

    symbols = preferred["YAHOO_SYMBOL"].dropna().unique().tolist()
    print(f"Downloading history for {len(symbols):,} symbols...")
    prices = yf.download(
        symbols,
        start=START_DATE,
        end=END_DATE,
        interval="1d",
        group_by="ticker",
        auto_adjust=False,
        actions=True,
        threads=True,
        progress=True,
    )

    records = []
    for symbol in symbols:
        try:
            frame = prices[symbol].dropna(how="all").reset_index()
            frame["YAHOO_SYMBOL"] = symbol
            records.append(frame)
        except (KeyError, TypeError):
            print(f"No history returned for {symbol}")

    if records:
        price_history = pd.concat(records, ignore_index=True)
        price_history = price_history.merge(
            preferred[["ISIN", "SYMBOL", "SEGMENT", "YAHOO_SYMBOL"]],
            on="YAHOO_SYMBOL",
            how="left",
        )
        price_history.to_parquet(OUTPUT_DIR / "daily_price_history.parquet", index=False)
        print(f"Saved {len(price_history):,} price rows.")
        display(price_history.head())
    else:
        print("No price history was returned.")
else:
    print("History download disabled. Set DOWNLOAD_HISTORY = True to run it.")

## Output files

- `nse_reference_master.csv`: normalized NSE reference rows.
- `isin_to_symbol.csv`: one preferred symbol per universe ISIN.
- `unmapped_isins.csv`: universe ISINs without a symbol match.
- `daily_price_history.parquet`: optional historical data when enabled.